In [1]:
from pathlib import Path
from typing import Optional
import os
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, numpy as np
from utils import split_dataframe_by_prefix, remove_rows_containing_string_in_column, create_table_with_id_and_unique_label, name_csv_file

In [2]:
df_simule = pd.DataFrame(pd.read_csv(r"../../datas/bronze/dataset_brut.csv"))

In [3]:
df_simule.head()

,timestamp,machine_id,type_machine,secteur,type_metal,vitesse_rotation_nominal,courant_moteur_nominal,pression_hydraulique_nominal,statut_nominal,temp_base_moteur,...,iot_vitesse_rotation,iot_courant_moteur,iot_pression_hydraulique,iot_temperature,iot_vibration_rms,iot_vibration_peak,iot_charge_moteur,age_jours,age_virtuel_jours,RUL_jours
0,2024-08-01 00:00:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.808858,0.000000,48.191631,0.914319,1.245143,45.0,1200.000000,840.000000,0.793750
1,2024-08-01 00:00:30,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.070450,0.000000,48.062069,0.874214,1.380316,45.0,1200.000347,840.000347,0.793403
2,2024-08-01 00:01:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.052242,0.000000,48.706761,0.858751,1.230252,45.0,1200.000694,840.000694,0.793056
3,2024-08-01 00:01:30,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,20.036585,0.000000,48.484258,0.849300,1.275070,45.0,1200.001042,840.001042,0.792708
4,2024-08-01 00:02:00,MCH-001,CNC_5_Axes,Aéro,Aluminium,6000,20.0,0.0,Production,40.0,...,6000.0,19.829501,0.009466,48.398311,0.861642,1.243091,45.0,1200.001389,840.001389,0.792361


In [4]:
df_simule.info()

<class 'pandas.DataFrame'>
RangeIndex: 14716800 entries, 0 to 14716799
Data columns (total 22 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   timestamp                     str    
 1   machine_id                    str    
 2   type_machine                  str    
 3   secteur                       str    
 4   type_metal                    str    
 5   vitesse_rotation_nominal      int64  
 6   courant_moteur_nominal        float64
 7   pression_hydraulique_nominal  float64
 8   statut_nominal                str    
 9   temp_base_moteur              float64
 10  iot_statut_machine            str    
 11  label_gmao                    str    
 12  iot_vitesse_rotation          float64
 13  iot_courant_moteur            float64
 14  iot_pression_hydraulique      float64
 15  iot_temperature               float64
 16  iot_vibration_rms             float64
 17  iot_vibration_peak            float64
 18  iot_charge_moteur             f

In [5]:
df_simule["label_gmao"].unique()

<ArrowStringArray>
[                           'Sain',                       'Alerte_P4',
        'Maintenance_Correctif_P4',                      'Alerte_P10',
       'Maintenance_Correctif_P10',                       'Alerte_P1',
        'Maintenance_Correctif_P1', 'Maintenance_Preventif_PLANIFIEE',
                       'Alerte_P9',        'Maintenance_Correctif_P9',
                       'Alerte_P5',        'Maintenance_Correctif_P5',
                       'Alerte_P7',                       'Alerte_P8',
        'Maintenance_Correctif_P8',                       'Alerte_P2',
        'Maintenance_Correctif_P2',        'Maintenance_Correctif_P7',
                       'Alerte_P6',        'Maintenance_Correctif_P6']
Length: 20, dtype: str

In [6]:
df_maintenance = remove_rows_containing_string_in_column(
    df=df_simule,
    column_name="label_gmao",
    string_to_remove="Sain"
)

In [7]:
df_maintenance["label_gmao"].unique()

<ArrowStringArray>
[                      'Alerte_P4',        'Maintenance_Correctif_P4',
                      'Alerte_P10',       'Maintenance_Correctif_P10',
                       'Alerte_P1',        'Maintenance_Correctif_P1',
 'Maintenance_Preventif_PLANIFIEE',                       'Alerte_P9',
        'Maintenance_Correctif_P9',                       'Alerte_P5',
        'Maintenance_Correctif_P5',                       'Alerte_P7',
                       'Alerte_P8',        'Maintenance_Correctif_P8',
                       'Alerte_P2',        'Maintenance_Correctif_P2',
        'Maintenance_Correctif_P7',                       'Alerte_P6',
        'Maintenance_Correctif_P6']
Length: 19, dtype: str

In [8]:
df_maintenance[["timestamp", "machine_id", "label_gmao"]].head()

,timestamp,machine_id,label_gmao
0,2024-08-01 11:09:00,MCH-001,Alerte_P4
1,2024-08-01 11:09:30,MCH-001,Alerte_P4
2,2024-08-01 11:10:00,MCH-001,Alerte_P4
3,2024-08-01 11:10:30,MCH-001,Alerte_P4
4,2024-08-01 11:11:00,MCH-001,Alerte_P4


# A faire
- debut panne, fin_panne, id_panne, id machine
- debut alerte, fin_panne, id_alerte, id machine
- table alerte
- table panne

In [9]:
df_alerte_splited, df_maintenance_splited = split_dataframe_by_prefix(df_maintenance, "label_gmao","Alerte")

In [10]:
df_maintenance_unique = create_table_with_id_and_unique_label(df_maintenance_splited,"label_gmao")
df_alerte_unique = create_table_with_id_and_unique_label(df_alerte_splited,"label_gmao")

In [11]:
df_maintenance_unique.head()

,label_gmao,id
949,Maintenance_Correctif_P4,4
1417,Maintenance_Correctif_P10,2
2154,Maintenance_Correctif_P1,1
2218,Maintenance_Preventif_PLANIFIEE,10
2462,Maintenance_Correctif_P9,9


In [12]:
folder_path = Path("../../datas/gold")
df_maintenance_unique.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="maintenance",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)
df_alerte_unique.to_csv(
    name_csv_file(
        folder_path=folder_path,
        filename="alerte",
        extension=".csv",
        type_dst="postgres"
    ), index=False, encoding='utf-8'
)